# 10 — Prompt-Fix Evaluation: Vorher/Nachher (Phase 3)

**Ziel:** Dominante Fehlerklassen aus Taxonomie (NB 09) per Prompt-Fix adressieren,
dann RA/SA/VA auf demselben n=20-Sample neu messen und Differenz mit Bootstrap-CI berichten.

**Fehlerklassen:**
- **C — yr-Vorzeichenfehler** (5 Fälle): LLM beschreibt `yr=0` (2011) als Wachstumsmerkmal
  trotz negativem SHAP-Beitrag. Fix: explizite Vorzeichen-Bindung in allen Prompts.
- **B2 — Nahe-Beitrag-Rangswap** (4 Fälle): LLM tauscht Features mit ähnlichem |Beitrag| um.
  Fix: explizite Rang-Bindung in allen Prompts.

**Pipeline:** Neue Erklärungen → gleicher NB08-Extraktor → RA/SA/VA → Bootstrap-CI Δ.

## 0 · Setup

In [ ]:
from __future__ import annotations

import sys, json, re, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib
import shap

from utils import INSTANCE_IDS, EXPLANATIONS_DIR, RESULTS_DIR, MODELS_DIR, PROMPTS_DIR
from utils.data import load_train_test
from utils.models import load_models
from utils.explanations import FEATURE_SCHEMA
from utils.tools import ToolBox, TOOL_DEFINITIONS
from utils.llm import ask_text, ask_with_images, DEFAULT_MODEL, _get_client, _with_retry

LOSS_KEY   = 'poisson_log'
MODEL      = DEFAULT_MODEL
MAX_TOKENS = 600
PLOTS_DIR  = EXPLANATIONS_DIR / 'plots'

# Ausgabe-Verzeichnisse für v2-Erklärungen
OUT04 = RESULTS_DIR / 'pipeline04_v2'
OUT05 = RESULTS_DIR / 'pipeline05_v2'
OUT06 = RESULTS_DIR / 'pipeline06_v2'
OUT_EVAL = RESULTS_DIR / 'eval08_ichmoukhamedov_v2'
for d in [OUT04, OUT05, OUT06, OUT_EVAL]:
    d.mkdir(parents=True, exist_ok=True)

PIPELINE_LABELS = {'04': 'JSON→Text', '05': 'Vision', '06': 'Tool-Use'}
XAI_MODELS = ['xgb', 'ebm']
TOP_K = 4

print(f'Modell:      {MODEL}')
print(f'Instanzen:   {INSTANCE_IDS}')
print(f'Gesamt:      {len(INSTANCE_IDS)} × {len(XAI_MODELS)} × 3 Pipelines = '
      f'{len(INSTANCE_IDS)*len(XAI_MODELS)*3} Erklärungen')

## 1 · Vorher-Ergebnisse laden (Baseline aus NB 08)

In [ ]:
before_path = RESULTS_DIR / 'eval08_ichmoukhamedov' / 'faithfulness_metrics.csv'
before_df = pd.read_csv(before_path)

print('Vorher-Daten (NB 08, originale Prompts):')
print(f'  {len(before_df)} Einträge')
display(before_df.groupby('pipeline_label')[['RA', 'SA', 'VA']].mean().round(3))

## 2 · Hilfsfunktionen (gemeinsam für alle Pipelines)

In [ ]:
WEEKDAYS = ['Sonntag', 'Montag', 'Dienstag', 'Mittwoch', 'Donnerstag', 'Freitag', 'Samstag']
MONTHS   = ['', 'Januar', 'Februar', 'März', 'April', 'Mai', 'Juni',
             'Juli', 'August', 'September', 'Oktober', 'November', 'Dezember']
WEATHER  = {1: 'klar/wenige Wolken', 2: 'Nebel/bewölkt',
             3: 'leichter Regen/Schnee', 4: 'Starkregen/Gewitter'}


def load_local(model_name: str, iid: int) -> dict:
    p = EXPLANATIONS_DIR / f'local_{model_name}_{LOSS_KEY}_inst{iid}.json'
    return json.loads(p.read_text())


def load_global(model_name: str) -> dict:
    p = EXPLANATIONS_DIR / f'global_{model_name}_{LOSS_KEY}.json'
    return json.loads(p.read_text())


def feature_value_summary(fv: dict) -> str:
    yr_str = '2011' if int(fv['yr']) == 0 else '2012'
    return (
        f"Uhrzeit: {int(fv['hr']):02d}:00 | Wochentag: {WEEKDAYS[int(fv['weekday'])]} | "
        f"Monat: {MONTHS[int(fv['mnth'])]} | Jahr: {yr_str} | "
        f"Wetter: {WEATHER.get(int(fv['weathersit']), fv['weathersit'])} | "
        f"Temp: ~{float(fv['temp'])*41:.1f}°C | "
        f"Luftf.: {float(fv['hum'])*100:.0f}% | "
        f"Wind: {float(fv['windspeed'])*67:.1f} km/h"
    )


print('Hilfsfunktionen geladen.')

## 3 · Pipeline 04 — JSON→Text (v2, fixierter Prompt)

In [ ]:
SYSTEM04 = (PROMPTS_DIR / 'pipeline_04_json.md').read_text()
print(f'System-Prompt P04: {len(SYSTEM04)} Zeichen')
# Sicherstellen, dass der Fix enthalten ist
assert 'Vorzeichen bindend' in SYSTEM04, 'Fix nicht gefunden!'
assert 'Rang bindend' in SYSTEM04, 'Fix nicht gefunden!'
print('✓ Vorzeichen- und Rang-Fix bestätigt.')

In [ ]:
def build_p04_user_prompt(model_name: str, iid: int) -> str:
    g = load_global(model_name)
    l = load_local(model_name, iid)
    fv = l['feature_values']

    top_global = sorted(g['importances'], key=lambda x: -x['importance'])[:5]
    top_local  = sorted(l['contributions'], key=lambda x: -abs(x['contribution']))[:TOP_K]

    global_str = '\n'.join(
        f"  {i+1}. {r['feature']}: {r['importance']:.4f}"
        for i, r in enumerate(top_global)
    )
    local_str = '\n'.join(
        f"  Rang {i}: {r['feature']} = {r['value']} → Beitrag {r['contribution']:+.4f}"
        for i, r in enumerate(top_local)
    )

    return (
        f"Modell: {model_name.upper()} | Instanz-ID: {iid}\n"
        f"{feature_value_summary(fv)}\n\n"
        f"Vorhersage (exp-Raum): {l['prediction']:.1f}  |  Tatsächlich: {l['y_true']:.0f}\n"
        f"Basiswert (Log-Raum): {l['base_value']:.4f}\n\n"
        f"Top-5 globale Feature-Wichtigkeiten:\n{global_str}\n\n"
        f"Top-{TOP_K} lokale SHAP-Beiträge (nach |Beitrag|, STRIKT absteigend sortiert):\n{local_str}\n"
    )


print('User-Prompt-Builder P04 definiert.')

In [ ]:
cache04 = {f.stem: json.loads(f.read_text()) for f in sorted(OUT04.glob('*.json'))}
missing04 = [
    (mn, iid) for mn in XAI_MODELS for iid in INSTANCE_IDS
    if f'{mn}_inst{iid}' not in cache04
]
print(f'P04 Cache: {len(cache04)} vorhanden, {len(missing04)} fehlend.')

total_in04, total_out04 = 0, 0
for model_name, iid in missing04:
    user_prompt = build_p04_user_prompt(model_name, iid)
    t0 = time.time()
    response = ask_text(
        user_prompt, system=SYSTEM04, model=MODEL,
        max_tokens=MAX_TOKENS, cache_system=True,
    )
    elapsed = time.time() - t0
    text  = response['content'][0]['text']
    usage = response.get('usage', {})
    in_t  = usage.get('input_tokens', 0)
    out_t = usage.get('output_tokens', 0)
    total_in04 += in_t; total_out04 += out_t

    l = load_local(model_name, iid)
    rec = {
        'pipeline': '04_json_v2', 'llm_model': MODEL, 'loss_key': LOSS_KEY,
        'xai_model': model_name, 'instance_id': iid, 'explanation': text,
        'elapsed_s': round(elapsed, 2),
        'usage': {'input_tokens': in_t, 'output_tokens': out_t},
        'prediction': l['prediction'], 'y_true': l['y_true'],
    }
    key = f'{model_name}_inst{iid}'
    (OUT04 / f'{key}.json').write_text(json.dumps(rec, indent=2, ensure_ascii=False))
    cache04[key] = rec
    print(f'  P04 {model_name.upper()} inst={iid}: in={in_t} out={out_t} t={elapsed:.1f}s')

if missing04:
    print(f'\nP04 gesamt: input={total_in04} output={total_out04}')
print(f'P04 v2: {len(cache04)} Erklärungen verfügbar.')

## 4 · Pipeline 05 — Vision (v2, fixierter Prompt)

In [ ]:
SYSTEM05 = (PROMPTS_DIR / 'pipeline_05_vision.md').read_text()
print(f'System-Prompt P05: {len(SYSTEM05)} Zeichen')
assert 'Vorzeichen bindend' in SYSTEM05, 'Fix nicht gefunden!'
assert 'Rang bindend' in SYSTEM05, 'Fix nicht gefunden!'
print('✓ Vorzeichen- und Rang-Fix bestätigt.')

In [ ]:
def build_p05_user_prompt(model_name: str, iid: int) -> str:
    l = load_local(model_name, iid)
    fv = l['feature_values']
    yr_str = '2011' if int(fv['yr']) == 0 else '2012'
    lines = [
        f'Modell: {model_name.upper()}',
        f'Instanz-ID: {iid}',
        f'Uhrzeit: {int(fv["hr"]):02d}:00 Uhr',
        f'Wochentag: {WEEKDAYS[int(fv["weekday"])]}',
        f'Monat: {MONTHS[int(fv["mnth"])]}',
        f'Jahr: {yr_str} (yr={int(fv["yr"])})',
        f'Wetter: {WEATHER.get(int(fv["weathersit"]), fv["weathersit"])}',
        f'Temperatur: ~{float(fv["temp"])*41:.1f} °C (normalisiert: {float(fv["temp"]):.2f})',
        f'Luftfeuchtigkeit: {float(fv["hum"])*100:.0f} %',
        f'Windgeschwindigkeit: {float(fv["windspeed"])*67:.1f} km/h',
        f'Feiertag: {"ja" if int(fv["holiday"]) == 1 else "nein"}',
        f'Tatsächliche Ausleihe: {int(l["y_true"])} Fahrräder',
        f'Modellvorhersage: {l["prediction"]:.1f} Fahrräder',
        '',
        'Bitte erkläre den Waterfall-Plot oben für diese Situation.',
    ]
    return '\n'.join(lines)


print('User-Prompt-Builder P05 definiert.')

In [ ]:
cache05 = {f.stem: json.loads(f.read_text()) for f in sorted(OUT05.glob('*.json'))}
missing05 = [
    (mn, iid) for mn in XAI_MODELS for iid in INSTANCE_IDS
    if f'{mn}_inst{iid}' not in cache05
]
print(f'P05 Cache: {len(cache05)} vorhanden, {len(missing05)} fehlend.')

total_in05, total_out05 = 0, 0
for model_name, iid in missing05:
    plot_path = PLOTS_DIR / f'waterfall_{model_name}_{LOSS_KEY}_inst{iid}.png'
    if not plot_path.exists():
        print(f'  [WARN] Plot fehlt: {plot_path} — überspringe.')
        continue
    user_prompt = build_p05_user_prompt(model_name, iid)
    t0 = time.time()
    response = ask_with_images(
        user_prompt, image_paths=[plot_path], system=SYSTEM05,
        model=MODEL, max_tokens=MAX_TOKENS, cache_system=True,
    )
    elapsed = time.time() - t0
    text  = response['content'][0]['text']
    usage = response.get('usage', {})
    in_t  = usage.get('input_tokens', 0)
    out_t = usage.get('output_tokens', 0)
    total_in05 += in_t; total_out05 += out_t

    l = load_local(model_name, iid)
    rec = {
        'pipeline': '05_vision_v2', 'llm_model': MODEL, 'loss_key': LOSS_KEY,
        'xai_model': model_name, 'instance_id': iid, 'explanation': text,
        'plot_file': plot_path.name, 'elapsed_s': round(elapsed, 2),
        'usage': {'input_tokens': in_t, 'output_tokens': out_t},
        'prediction': l['prediction'], 'y_true': l['y_true'],
    }
    key = f'{model_name}_inst{iid}'
    (OUT05 / f'{key}.json').write_text(json.dumps(rec, indent=2, ensure_ascii=False))
    cache05[key] = rec
    print(f'  P05 {model_name.upper()} inst={iid}: in={in_t} out={out_t} t={elapsed:.1f}s')

if missing05:
    print(f'\nP05 gesamt: input={total_in05} output={total_out05}')
print(f'P05 v2: {len(cache05)} Erklärungen verfügbar.')

## 5 · Pipeline 06 — Tool-Use (v2, fixierter Prompt)

In [ ]:
SYSTEM06_TPL = (PROMPTS_DIR / 'pipeline_06_tooluse.md').read_text()
print(f'System-Prompt P06: {len(SYSTEM06_TPL)} Zeichen')
assert 'Vorzeichen bindend' in SYSTEM06_TPL, 'Fix nicht gefunden!'
assert 'Rang bindend' in SYSTEM06_TPL, 'Fix nicht gefunden!'
print('✓ Vorzeichen- und Rang-Fix bestätigt.')

X_train, y_train, X_test, y_test = load_train_test()
xgb_model, ebm_model = load_models(LOSS_KEY)
print(f'Modelle geladen: XGB + EBM ({LOSS_KEY})')
MAX_TOKENS_TOOL = 1200

In [ ]:
def tool_use_loop_v2(
    client,
    toolbox: ToolBox,
    user_message: str,
    system: str,
    max_rounds: int = 10,
) -> tuple[str, list, int, int, str]:
    messages = [{'role': 'user', 'content': user_message}]
    total_in, total_out = 0, 0
    last_text = ''
    final_stop_reason = 'max_rounds'

    for round_num in range(max_rounds):
        response = _with_retry(
            client.messages.create,
            model=MODEL,
            max_tokens=MAX_TOKENS_TOOL,
            system=[{'type': 'text', 'text': system,
                     'cache_control': {'type': 'ephemeral'}}],
            tools=TOOL_DEFINITIONS,
            messages=messages,
        )
        usage = response.usage
        total_in  += usage.input_tokens
        total_out += usage.output_tokens
        messages.append({'role': 'assistant', 'content': response.content})

        candidate = next(
            (b.text for b in response.content if hasattr(b, 'text') and b.text), ''
        )
        if candidate:
            last_text = candidate

        if response.stop_reason == 'end_turn':
            return last_text, toolbox.call_log, total_in, total_out, 'end_turn'

        if response.stop_reason != 'tool_use':
            final_stop_reason = response.stop_reason
            break

        tool_results = []
        for block in response.content:
            if block.type != 'tool_use':
                continue
            result = toolbox.dispatch(block.name, block.input)
            tool_results.append({
                'type': 'tool_result',
                'tool_use_id': block.id,
                'content': json.dumps(result, ensure_ascii=False),
            })
            print(f'    [{round_num+1}] {block.name}({list(block.input.keys())}) -> ok')
        messages.append({'role': 'user', 'content': tool_results})

    print(f'  [WARN] stop_reason={final_stop_reason!r} – verwende letzten Teiltext.')
    return last_text or '[Keine Antwort]', toolbox.call_log, total_in, total_out, final_stop_reason


print('Tool-Use-Schleife (v2) definiert.')

In [ ]:
def build_p06_task_prompt(model_name: str, iid: int) -> str:
    row = X_test.iloc[iid]
    y   = float(y_test.iloc[iid])
    yr_str = '2011' if int(row['yr']) == 0 else '2012'
    return (
        f'Erkläre die Modellvorhersage für Instanz {iid} des {model_name.upper()}-Modells.\n'
        f'Tatsächliche Ausleihe: {y:.0f} Fahrräder.\n'
        f'Uhrzeit: {int(row["hr"]):02d}:00 | Wochentag: {WEEKDAYS[int(row["weekday"])]} | '
        f'Monat: {MONTHS[int(row["mnth"])]} | Jahr: {yr_str} (yr={int(row["yr"])}) | '
        f'Wetter: {WEATHER.get(int(row["weathersit"]), row["weathersit"])}.\n'
        f'Verwende die verfügbaren Tools für eine vollständige Analyse.'
    )


print('Task-Prompt-Builder P06 definiert.')

In [ ]:
cache06 = {f.stem: json.loads(f.read_text()) for f in sorted(OUT06.glob('*.json'))}
missing06 = [
    (mn, iid) for mn in XAI_MODELS for iid in INSTANCE_IDS
    if f'{mn}_inst{iid}' not in cache06
]
print(f'P06 Cache: {len(cache06)} vorhanden, {len(missing06)} fehlend.')

client = _get_client()
total_in06, total_out06 = 0, 0

model_obj_map = {'xgb': xgb_model, 'ebm': ebm_model}

for model_name, iid in missing06:
    model_obj = model_obj_map[model_name]
    toolbox   = ToolBox(model_obj, X_train, X_test, y_test, model_name)
    prompt    = build_p06_task_prompt(model_name, iid)
    system    = SYSTEM06_TPL.replace('{model_name}', model_name.upper())

    print(f'\nP06 {model_name.upper()} inst={iid}')
    t0 = time.time()
    try:
        text, call_log, in_t, out_t, stop_r = tool_use_loop_v2(
            client, toolbox, prompt, system
        )
    except Exception as e:
        print(f'  [ERROR] {e}')
        continue
    elapsed = time.time() - t0
    total_in06 += in_t; total_out06 += out_t

    l = load_local(model_name, iid)
    rec = {
        'pipeline': '06_tooluse_v2', 'llm_model': MODEL, 'loss_key': LOSS_KEY,
        'xai_model': model_name, 'instance_id': iid, 'explanation': text,
        'n_tool_calls': len(call_log), 'tool_calls': call_log,
        'elapsed_s': round(elapsed, 2), 'stop_reason': stop_r,
        'usage': {'input_tokens': in_t, 'output_tokens': out_t},
        'prediction': l['prediction'], 'y_true': l['y_true'],
    }
    key = f'{model_name}_inst{iid}'
    (OUT06 / f'{key}.json').write_text(json.dumps(rec, indent=2, ensure_ascii=False))
    cache06[key] = rec
    print(f'  → in={in_t} out={out_t} tools={len(call_log)} t={elapsed:.1f}s stop={stop_r}')

if missing06:
    print(f'\nP06 gesamt: input={total_in06} output={total_out06}')
print(f'P06 v2: {len(cache06)} Erklärungen verfügbar.')

## 6 · Extraktion (RA/SA/VA) — gleiche Logik wie NB 08

In [ ]:
EXTRACTION_SYSTEM = (
    'Du bist ein Extraktionsmodell für XAI-Narrative eines Fahrradverleih-Modells.\n'
    'Extrahiere die angeforderten Informationen ausschließlich aus dem Narrativ.\n'
    'Antworte ausschließlich mit einem validen JSON-Objekt — kein Text, keine Markdown-Blöcke.'
)

_DENORM = {
    'temp':      lambda v: v * 41,
    'hum':       lambda v: v * 100,
    'windspeed': lambda v: v * 67,
}


def build_extraction_prompt(explanation: str, xai_model: str) -> str:
    feat_descs = {f: FEATURE_SCHEMA[f]['description'] for f in FEATURE_SCHEMA}
    payload = {
        'aufgabe': (
            f'Extrahiere strukturierte Informationen aus dem folgenden deutschen Narrativ '
            f'zu einem {xai_model}-Regressionsmodell für einen Fahrradverleih. '
            f'Das Modell sagt stündliche Fahrrad-Ausleihen voraus. '
            f'Positive Beiträge erhöhen die Vorhersage, negative senken sie.'
        ),
        'narrativ': explanation,
        'alle_features': list(FEATURE_SCHEMA.keys()),
        'feature_beschreibungen': feat_descs,
        'extraktionsanweisung': (
            'Gib für jedes Feature, das im Narrativ als wichtig erwähnt wird, ein Objekt zurück:\n'
            '  rank: 0-basierter Wichtigkeitsrang laut Narrativ (0 = wichtigstes Feature)\n'
            '  sign: +1 wenn das Feature die Vorhersage erhöht, -1 wenn es sie senkt\n'
            '  value: numerischer Featurewert, falls explizit im Narrativ genannt, sonst null\n'
            '  assumption: einziger Satz mit Hintergrundwissen warum das Feature diesen '
            'Einfluss hat; "None" falls kein Hintergrundwissen hinzugefügt wurde'
        ),
        'ausgabeformat_beispiel': {
            'hr':   {'rank': 0, 'sign': 1, 'value': 13,   'assumption': 'Mittagszeit ist nachfragereich.'},
            'temp': {'rank': 1, 'sign': -1, 'value': None, 'assumption': 'Kälte schreckt Radfahrer ab.'},
        },
    }
    return json.dumps(payload, ensure_ascii=False, indent=2)


def parse_extraction(raw: str) -> dict:
    m = re.search(r'\{.*\}', raw, re.DOTALL)
    if not m:
        return {}
    try:
        return json.loads(m.group())
    except json.JSONDecodeError:
        return {}


def is_value_match(feat: str, extracted: float, gt: float, tol: float = 1.0) -> bool:
    if abs(extracted - gt) <= tol:
        return True
    if feat in _DENORM:
        dv = _DENORM[feat](gt)
        if abs(extracted - dv) <= tol:
            return True
    return False


def compute_faithfulness(extraction: dict, gt_contributions: list) -> dict:
    gt_rank  = {c['feature']: i for i, c in enumerate(gt_contributions)}
    gt_sign  = {c['feature']: (1 if c['contribution'] >= 0 else -1) for c in gt_contributions}
    gt_value = {c['feature']: c['value'] for c in gt_contributions}

    ra_hits = ra_n = sa_hits = sa_n = va_hits = va_n = 0

    for feat, info in extraction.items():
        feat_key = feat.lower()
        if feat_key not in gt_rank:
            continue
        r = info.get('rank')
        if r is not None:
            ra_n += 1
            try:
                if int(float(r)) == gt_rank[feat_key]:
                    ra_hits += 1
            except (ValueError, TypeError):
                pass
        s = info.get('sign')
        if s is not None:
            sa_n += 1
            try:
                if int(float(s)) == gt_sign[feat_key]:
                    sa_hits += 1
            except (ValueError, TypeError):
                pass
        v = info.get('value')
        if v is not None and str(v).lower() not in ('null', 'none', ''):
            try:
                v_float = float(v)
                gt_v    = float(gt_value.get(feat_key, 0))
                va_n += 1
                if is_value_match(feat_key, v_float, gt_v):
                    va_hits += 1
            except (ValueError, TypeError):
                pass

    return {
        'RA': round(ra_hits / ra_n, 4) if ra_n > 0 else None,
        'SA': round(sa_hits / sa_n, 4) if sa_n > 0 else None,
        'VA': round(va_hits / va_n, 4) if va_n > 0 else None,
        'RA_hits': ra_hits, 'RA_n': ra_n,
        'SA_hits': sa_hits, 'SA_n': sa_n,
        'VA_hits': va_hits, 'VA_n': va_n,
    }


print('Extraktions-Funktionen geladen (identisch zu NB 08).')

In [ ]:
# Alle v2-Erklärungen in einen DataFrame laden
v2_records = []
for pipeline, cache, label in [
    ('04', cache04, 'JSON→Text'),
    ('05', cache05, 'Vision'),
    ('06', cache06, 'Tool-Use'),
]:
    for xai in XAI_MODELS:
        for iid in INSTANCE_IDS:
            key = f'{xai}_inst{iid}'
            if key not in cache:
                print(f'FEHLT: pipeline {pipeline}, {key}')
                continue
            d = cache[key]
            gt_path = EXPLANATIONS_DIR / f'local_{xai}_{LOSS_KEY}_inst{iid}.json'
            gt = json.loads(gt_path.read_text())
            v2_records.append({
                'pipeline':          pipeline,
                'pipeline_label':    label,
                'xai_model':         xai.upper(),
                'instance_id':       iid,
                'explanation':       d.get('explanation', ''),
                'gt_contributions':  gt['contributions'][:TOP_K],
            })

v2_df = pd.DataFrame(v2_records)
print(f'{len(v2_df)} v2-Narrative geladen.')

In [ ]:
CACHE_EXT_PATH = OUT_EVAL / 'extractions.json'

if CACHE_EXT_PATH.exists():
    extractions_v2 = json.loads(CACHE_EXT_PATH.read_text())
    print(f'Extractions aus Cache geladen: {len(extractions_v2)} Einträge.')
else:
    extractions_v2 = {}

missing_ext = [
    row for _, row in v2_df.iterrows()
    if f"{row['pipeline']}_{row['xai_model'].lower()}_inst{row['instance_id']}" not in extractions_v2
]
print(f'{len(missing_ext)} fehlende Extraktionen — starte API-Calls...')

total_ext_in, total_ext_out = 0, 0
for row in missing_ext:
    key = f"{row['pipeline']}_{row['xai_model'].lower()}_inst{row['instance_id']}"
    prompt = build_extraction_prompt(row['explanation'], row['xai_model'])
    response = ask_text(
        prompt, system=EXTRACTION_SYSTEM, model=MODEL,
        max_tokens=700, cache_system=True,
    )
    usage   = response.get('usage', {})
    in_tok  = usage.get('input_tokens', 0)
    out_tok = usage.get('output_tokens', 0)
    total_ext_in += in_tok; total_ext_out += out_tok
    raw        = response['content'][0]['text'].strip()
    extraction = parse_extraction(raw)
    extractions_v2[key] = {
        'pipeline_label': row['pipeline_label'],
        'xai_model':      row['xai_model'],
        'instance_id':    row['instance_id'],
        'raw':            raw,
        'extraction':     extraction,
        'usage':          {'input_tokens': in_tok, 'output_tokens': out_tok},
    }
    print(f'  {key}: {len(extraction)} Features  in={in_tok}  out={out_tok}')

if missing_ext:
    CACHE_EXT_PATH.write_text(json.dumps(extractions_v2, indent=2, ensure_ascii=False))
    print(f'Gespeichert: {CACHE_EXT_PATH}')
    print(f'Extraktion gesamt: input={total_ext_in}  output={total_ext_out}')
else:
    print('Alle Extraktionen bereits im Cache.')

In [ ]:
# Faithfulness-Metriken berechnen
faith_v2_rows = []
for _, row in v2_df.iterrows():
    key = f"{row['pipeline']}_{row['xai_model'].lower()}_inst{row['instance_id']}"
    ext = extractions_v2.get(key, {}).get('extraction', {})
    m   = compute_faithfulness(ext, row['gt_contributions'])
    faith_v2_rows.append({
        'pipeline_label': row['pipeline_label'],
        'xai_model':      row['xai_model'],
        'instance_id':    row['instance_id'],
        **m,
    })

faith_v2_df = pd.DataFrame(faith_v2_rows)

print('Nachher-Metriken (v2, fixierte Prompts):')
display(faith_v2_df.groupby('pipeline_label')[['RA', 'SA', 'VA']].mean().round(3))

faith_v2_df.to_csv(OUT_EVAL / 'faithfulness_metrics.csv', index=False)
print(f'Gespeichert: {OUT_EVAL / "faithfulness_metrics.csv"}')

## 7 · Vorher/Nachher-Vergleich mit Bootstrap-CI

In [ ]:
def bootstrap_ci(
    values: np.ndarray,
    n_boot: int = 2000,
    alpha: float = 0.05,
    rng: np.random.Generator | None = None,
) -> tuple[float, float, float]:
    """Bootstrap-Konfidenzintervall (BCa nicht nötig bei n=20; percentile reicht)."""
    if rng is None:
        rng = np.random.default_rng(42)
    vals = np.array([v for v in values if v is not None and not np.isnan(v)])
    if len(vals) == 0:
        return (np.nan, np.nan, np.nan)
    boot = rng.choice(vals, size=(n_boot, len(vals)), replace=True).mean(axis=1)
    lo, hi = np.percentile(boot, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(vals.mean()), float(lo), float(hi)


print('Bootstrap-CI-Funktion geladen.')

In [ ]:
PIPELINES_ORDER = ['JSON→Text', 'Vision', 'Tool-Use']
METRICS = ['RA', 'SA', 'VA']
rng = np.random.default_rng(42)

comparison_rows = []
for pl in PIPELINES_ORDER:
    b_sub = before_df[before_df.pipeline_label == pl]
    a_sub = faith_v2_df[faith_v2_df.pipeline_label == pl]
    for metric in METRICS:
        bvals = b_sub[metric].dropna().values
        avals = a_sub[metric].dropna().values
        b_m, b_lo, b_hi = bootstrap_ci(bvals, rng=rng)
        a_m, a_lo, a_hi = bootstrap_ci(avals, rng=rng)
        diff = a_m - b_m
        # Bootstrap-CI für die Differenz
        n_boot = 2000
        boot_b = rng.choice(bvals, size=(n_boot, len(bvals)), replace=True).mean(axis=1)
        boot_a = rng.choice(avals, size=(n_boot, len(avals)), replace=True).mean(axis=1)
        boot_diff = boot_a - boot_b
        d_lo, d_hi = np.percentile(boot_diff, [2.5, 97.5])
        comparison_rows.append({
            'Pipeline': pl, 'Metrik': metric,
            'Vorher_mean': round(b_m, 3), 'Vorher_CI95': f'[{b_lo:.3f}, {b_hi:.3f}]',
            'Nachher_mean': round(a_m, 3), 'Nachher_CI95': f'[{a_lo:.3f}, {a_hi:.3f}]',
            'Δ': round(diff, 3), 'Δ_CI95': f'[{d_lo:.3f}, {d_hi:.3f}]',
            'Δ_signifikant': 'JA' if d_lo > 0 else ('NEGATIV' if d_hi < 0 else 'n.s.'),
        })

comp_df = pd.DataFrame(comparison_rows)
print('=== Vorher/Nachher-Vergleich (95 %-Bootstrap-CI) ===')
display(comp_df.set_index(['Pipeline', 'Metrik']))

comp_df.to_csv(OUT_EVAL / 'comparison_before_after.csv', index=False)
print(f'\nGespeichert: {OUT_EVAL / "comparison_before_after.csv"}')

## 8 · Abbildung: Vorher/Nachher-Balkendiagramm

In [ ]:
metric_labels = {
    'RA': 'RA — Rang-Genauigkeit',
    'SA': 'SA — Vorzeichen-Genauigkeit',
    'VA': 'VA — Wert-Genauigkeit',
}
colors_before = ['#4c72b0', '#dd8452', '#55a868']
colors_after  = ['#1f4e9c', '#b85a1a', '#2d7a43']

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
x = np.arange(len(PIPELINES_ORDER))
w = 0.35

for ax, metric, title in zip(axes, METRICS, metric_labels.values()):
    before_vals = [
        before_df[before_df.pipeline_label == pl][metric].mean()
        for pl in PIPELINES_ORDER
    ]
    after_vals = [
        faith_v2_df[faith_v2_df.pipeline_label == pl][metric].mean()
        for pl in PIPELINES_ORDER
    ]
    b1 = ax.bar(x - w/2, before_vals, w, label='Vorher (v1)',
                color=colors_before, alpha=0.75)
    b2 = ax.bar(x + w/2, after_vals,  w, label='Nachher (v2)',
                color=colors_after, alpha=0.95)
    ax.set_xticks(x)
    ax.set_xticklabels([p.replace('→', '→\n') for p in PIPELINES_ORDER], fontsize=8)
    ax.set_ylim(0, 1.15)
    ax.set_title(title, fontsize=9, pad=8)
    ax.set_ylabel('Genauigkeit (0–1)')
    ax.axhline(1.0, color='gray', linestyle='--', alpha=0.35, linewidth=1)
    for bar, val in zip(list(b1) + list(b2), before_vals + after_vals):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7.5)

axes[0].legend(fontsize=8)
plt.suptitle(
    'Faithfulness-Metriken: Vorher/Nachher Prompt-Fix\n'
    '(C: yr-Vorzeichenbindung; B2: Rangbindung)',
    y=1.02, fontsize=11,
)
plt.tight_layout()
out_fig = RESULTS_DIR / 'eval10_prompt_fix_comparison.png'
plt.savefig(out_fig, dpi=130, bbox_inches='tight')
plt.show()
print(f'Gespeichert: {out_fig}')

## 9 · Zusammenfassung

In [ ]:
print('=== Phase 3 — Prompt-Fix Evaluation: Zusammenfassung ===')
print()
print('Prompt-Fixes (alle 3 Pipelines):')
print('  1. yr-Vorzeichen-Bindung: yr=0 (2011) → negativer Beitrag (explizit im Prompt)')
print('  2. Rang-Bindung: Features in absteigender |Beitrag|-Reihenfolge nennen')
print()

print('Vorher/Nachher (Ø über alle Pipelines):')
for metric in METRICS:
    b_all = before_df[metric].dropna().mean()
    a_all = faith_v2_df[metric].dropna().mean()
    delta = a_all - b_all
    sig_rows = comp_df[(comp_df.Metrik == metric) & (comp_df['Δ_signifikant'] == 'JA')]
    sig_str = f'{len(sig_rows)}/3 Pipelines signifikant (p<0.05)' if len(sig_rows) > 0 else 'keine signif.'
    print(f'  {metric}: {b_all:.3f} → {a_all:.3f}  (Δ={delta:+.3f})  [{sig_str}]')

print()
print('Artefakte:')
print(f'  Erklärungen v2: {OUT04}  {OUT05}  {OUT06}')
print(f'  Extraktion v2:  {OUT_EVAL / "extractions.json"}')
print(f'  Metriken v2:    {OUT_EVAL / "faithfulness_metrics.csv"}')
print(f'  Vergleich:      {OUT_EVAL / "comparison_before_after.csv"}')
print(f'  Abbildung:      {RESULTS_DIR / "eval10_prompt_fix_comparison.png"}')